<a href="https://colab.research.google.com/github/GaganDeshbhandari/ML-major-project-notebook/blob/main/factllama_qlora_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
# Cell 1
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
# Cell 2
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )
else:
    print("No GPU detected. Select a GPU runtime.")


PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [2]:
# Cell 3
%pip install -q -U transformers datasets peft trl accelerate bitsandbytes sentencepiece huggingface_hub


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 11.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.


In [23]:
# Cell 4
import torch
import transformers
import datasets
import peft
import trl
import accelerate
import bitsandbytes
import huggingface_hub

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

print("\nInstalled packages:")
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("Accelerate:", accelerate.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4

Installed packages:
Transformers: 5.19.0
Datasets: 5.1.0
PEFT: 0.21.2
TRL: 1.15.0
Accelerate: 1.15.0
BitsAndBytes: 0.50.2


In [27]:
# Cell 5
# Run this only one time to link the system to the Hugging-Face
from huggingface_hub import login
login()

In [28]:
# Cell 6
from huggingface_hub import model_info

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"

info = model_info(MODEL_ID)

print("Model:", info.id)
print("Model access verified.")

Model: meta-llama/Llama-3.2-3B-Instruct
Model access verified.


In [25]:
# Cell 7
from pathlib import Path

DATASET_ROOT = Path(
    "/content/drive/MyDrive/FactLLaMA Datasets"
)

RAWFC_DIR = DATASET_ROOT / "RAWFC"
LIAR_RAW_DIR = DATASET_ROOT / "LIAR-RAW"
DATA_DIR = DATASET_ROOT / "factllama_data"

print("RAWFC extracted:", RAWFC_DIR.exists())
print("LIAR-RAW extracted:", LIAR_RAW_DIR.exists())
print("Processed data:", DATA_DIR.exists())

for split in ["train", "val", "test"]:
    path = DATA_DIR / f"{split}.jsonl"
    print(f"{split}: exists={path.exists()}")

RAWFC extracted: True
LIAR-RAW extracted: True
Processed data: True
train: exists=True
val: exists=True
test: exists=True


In [29]:
# Cell 8
import json
from pathlib import Path
from transformers import AutoTokenizer

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"
DATA_DIR = DATASET_ROOT / "factllama_data"

# Load the tokenizer; this also checks model-repository access.
tok_loader = AutoTokenizer
tokenizer = tok_loader.from_pretrained(MODEL_ID)

SYSTEM_PROMPT = (
    "You are a fact-checking assistant. Evaluate the claim using the "
    "provided evidence. Return exactly one label: true, half, or false. "
    "Do not use outside knowledge."
)

def build_prompt(record, max_evidence=None):
    evidence = record["evidence"]
    if max_evidence is not None:
        evidence = evidence[:max_evidence]

    evidence_text = "\n".join(
        f"[{i + 1}] {item['text']}"
        for i, item in enumerate(evidence)
    )

    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": (
                f"Claim: {record['claim']}\n\n"
                f"Evidence:\n{evidence_text}\n\n"
                "Verdict:"
            )
        }
    ]

def count_tokens(messages):
    rendered = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    return len(tokenizer.encode(rendered, add_special_tokens=False))

with open(DATA_DIR / "train.jsonl", encoding="utf-8") as f:
    train_records = [json.loads(line) for line in f]

all_evidence_lengths = []
capped_lengths = []

for record in train_records:
    all_evidence_lengths.append(
        count_tokens(build_prompt(record))
    )
    capped_lengths.append(
        count_tokens(build_prompt(record, max_evidence=12))
    )

def summarize(name, values):
    values = sorted(values)
    n = len(values)
    print(f"\n{name}")
    print("Minimum:", values[0])
    print("Median:", values[n // 2])
    print("90th percentile:", values[int(n * 0.90)])
    print("Maximum:", values[-1])
    print("Prompts over 2048 tokens:", sum(x > 2048 for x in values))
    print("Prompts over 4096 tokens:", sum(x > 4096 for x in values))

print("Tokenizer loaded successfully:", MODEL_ID)
summarize("ALL EVIDENCE", all_evidence_lengths)
summarize("FIRST 12 EVIDENCE SENTENCES", capped_lengths)

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Tokenizer loaded successfully: meta-llama/Llama-3.2-3B-Instruct

ALL EVIDENCE
Minimum: 189
Median: 1317
90th percentile: 3135
Maximum: 5296
Prompts over 2048 tokens: 455
Prompts over 4096 tokens: 48

FIRST 12 EVIDENCE SENTENCES
Minimum: 189
Median: 595
90th percentile: 750
Maximum: 3082
Prompts over 2048 tokens: 1
Prompts over 4096 tokens: 0


In [30]:
# Cell 9
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"

assert torch.cuda.is_available(), "CUDA GPU is not available. Check Colab runtime type."

print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory:", round(
    torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
), "GB")

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    dtype=torch.float16,
    device_map="auto",
)

model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("\nModel loaded and LoRA adapters configured.")
print("Allocated GPU memory:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

GPU: Tesla T4
GPU memory: 14.56 GB


model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511

Model loaded and LoRA adapters configured.
Allocated GPU memory: 2.91 GB


In [31]:
# Cell 10
import json
from pathlib import Path
from datasets import Dataset

DATA_DIR = DATASET_ROOT / "factllama_data"
MAX_LENGTH = 2048
MAX_EVIDENCE = 12
VALID_LABELS = {"true", "half", "false"}

SYSTEM_PROMPT = (
    "You are a fact-checking assistant. Evaluate the claim using only "
    "the provided evidence. Output exactly one verdict: true, half, or false."
)

def encode_record(record):
    label = record["label"].strip().lower()
    if label not in VALID_LABELS:
        raise ValueError(f"Unexpected label: {label}")

    evidence = record["evidence"][:MAX_EVIDENCE]
    evidence_text = "\n".join(
        f"[{i + 1}] {item['text']}" for i, item in enumerate(evidence)
    )

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": (
                f"Claim: {record['claim']}\n\n"
                f"Evidence:\n{evidence_text}\n\n"
                "Verdict:"
            ),
        },
    ]

    # Prompt ends immediately before the assistant's verdict.
    prompt_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
    )

    # The desired assistant output is only the normalized verdict.
    target_ids = tokenizer.encode(
        label, add_special_tokens=False
    ) + [tokenizer.eos_token_id]

    if len(target_ids) >= MAX_LENGTH:
        raise ValueError("Target unexpectedly exceeds the context limit.")

    # Preserve the beginning of the prompt and always retain the target.
    max_prompt_length = MAX_LENGTH - len(target_ids)
    prompt_ids = prompt_ids[:max_prompt_length]

    input_ids = prompt_ids + target_ids
    labels = [-100] * len(prompt_ids) + target_ids

    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": labels,
        "length": len(input_ids),
        "verdict": label,
    }

encoded_datasets = {}

for split in ["train", "val", "test"]:
    path = DATA_DIR / f"{split}.jsonl"

    with path.open(encoding="utf-8") as f:
        records = [json.loads(line) for line in f]

    encoded = [encode_record(record) for record in records]
    dataset = Dataset.from_list(encoded)
    encoded_datasets[split] = dataset

    print(f"\n{split.upper()}")
    print("Examples:", len(dataset))
    print("Label distribution:", {
        label: sum(x["verdict"] == label for x in encoded)
        for label in sorted(VALID_LABELS)
    })
    print("Maximum encoded length:", max(x["length"] for x in encoded))
    print("Examples at context limit:",
          sum(x["length"] == MAX_LENGTH for x in encoded))

# Check that prompt tokens are masked and target tokens are supervised.
sample = encoded_datasets["train"][0]
supervised = [x for x in sample["labels"] if x != -100]

assert len(supervised) > 0
assert len(sample["input_ids"]) <= MAX_LENGTH
assert all(x == -100 for x in sample["labels"][:-
    len(supervised)])
assert tokenizer.decode(supervised, skip_special_tokens=True).strip() in VALID_LABELS

print("\nPreprocessing checks passed.")
print("Decoded sample target:",
      tokenizer.decode(supervised, skip_special_tokens=True))

OverflowError: There was an overflow with type <class 'list'>. Try to reduce writer_batch_size to have batches smaller than 2GB.
(Could not convert Encoding(num_tokens=399, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing]) with type tokenizers.Encoding: did not recognize Python value type when inferring an Arrow data type)